# When can a model tell it doesn't know? A toy model of gating

This notebook reproduces, end to end, the central result behind *physics-aware
abstention*: **a frozen linear probe on a network's activations can let the model
restrain itself — but only under a specific, checkable condition.**

It runs in a couple of minutes on CPU. Read the markdown, run the cells top to
bottom, and you'll reproduce the four-panel figure and understand exactly what it
means.

---

## The idea in one paragraph

We train a small network to predict a number `y` from an input `x`. Sometimes `x`
is **missing information that `y` genuinely depends on** — so no correct answer
exists (the problem is *unanswerable*). We freeze the network and ask a simple
linear probe, reading only the network's internal activations, to flag those
unanswerable inputs. If it can, we can **gate**: let the model answer when it's
confident and abstain when it isn't. The question this notebook settles is *when
that works and when it can't possibly work.*

## 1. The instrument: a problem with a tunable 'identifiability gap'

We use the simplest possible setup where we control everything. Latent variables
`z` are split into two blocks:

- a **core** block, always observed;
- a **peripheral** block, sometimes corrupted.

The target mixes them, with a knob `f` setting how much of `y` rides on the
peripheral block:

$$ y = \sqrt{1-f}\,(w_{\text{core}}\cdot z_{\text{core}}) + \sqrt{f}\,(w_{\text{periph}}\cdot z_{\text{periph}}) + \varepsilon $$

When the peripheral block is intact, the input is **answerable**. We break it two
ways, at the *same* `f` (so the resulting error is the same size):

- **zeroed** — set the peripheral block to 0. The corruption is *visible in the
  input* (those coordinates are exactly zero). This is the 'missing sensor' case.
- **resampled** — replace it with a fresh random draw. The input still looks
  perfectly normal; only its *value* is now wrong. This is the
  'subtly-wrong-but-complete' case.

The **identifiability gap** `γ` is how much bigger the irreducible error is on an
unanswerable input than on an answerable one. `γ = 1` means fully answerable;
large `γ` means deeply unanswerable. Crucially, we can compute `γ` exactly.

In [ ]:
import sys
from pathlib import Path
# make the analysis/ modules importable whether run from repo root or analysis/
for p in ('analysis', '.', '..'):
    if (Path(p) / 'toy_identifiability.py').exists():
        sys.path.insert(0, str(Path(p).resolve())); break

import numpy as np
import matplotlib.pyplot as plt
from toy_identifiability import make_instrument, generate, gamma_of

inst = make_instrument(k_core=8, k_periph=8, sigma=0.1, seed=0)
print('core latents:', inst.k_core, ' peripheral latents:', inst.k_periph)

# Show the gap we can dial:
for f in [0.0, 0.2, 0.5, 0.8]:
    print(f'f={f:.1f}  ->  gamma(zeroed)={gamma_of(f, inst.sigma, "zeroed"):6.1f}')

### What the corruptions look like

Let's confirm the two corruptions do what we claim: *zeroed* wipes the peripheral
block to zero (visible), *resampled* keeps it looking in-distribution (invisible).

In [ ]:
f = 0.6
clean = generate(inst, 300, f, 'clean', seed=1)
zeroed = generate(inst, 300, f, 'zeroed', seed=1)
resampled = generate(inst, 300, f, 'resampled', seed=1)
kc = inst.k_core

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (name, d) in zip(axes, [('answerable (clean)', clean), ('zeroed', zeroed), ('resampled', resampled)]):
    periph = d['x'][:, kc:]
    ax.hist(periph.ravel(), bins=40, color='#2a78d6', alpha=0.8)
    ax.set_title(f'{name}\nperipheral input values'); ax.set_xlabel('value')
axes[0].set_ylabel('count')
plt.tight_layout(); plt.show()
print('Note: zeroed is a spike at 0 (visible); resampled looks like clean (invisible).')

## 2. Train the readout, then freeze it

We train a small MLP to predict `y` from `x` on **answerable data only** — it
never sees a corrupted input during training, and never gets an 'is this
answerable' label. Everything about abstention has to come from reading its frozen
activations afterward.

In [ ]:
import torch
from toy_identifiability import Readout, train_readout, evaluate

f = 0.6
train = generate(inst, 3000, f, 'clean', seed=1)
model = Readout(inst.k_core + inst.k_periph, hidden=64)
train_readout(model, train, epochs=250, seed=0)

# sanity: it predicts answerable y well, and does worse on corrupted inputs
for name in ['clean', 'zeroed', 'resampled']:
    ev = evaluate(model, generate(inst, 800, f, name, seed=7))
    print(f'{name:10s} mean squared error = {ev["loss"].mean():.3f}')

## 3. The probe, and the two questions

We fit a **linear probe** on the frozen activations to predict *answerable vs
unanswerable*. Then we ask two different questions that turn out to have different
answers:

1. **Detection** — can the probe tell answerable from unanswerable? (AUROC)
2. **Benefit** — does gating on the probe actually reduce error on what we keep?
   (how much of the *oracle's* selective-risk reduction it captures)

In [ ]:
from toy_identifiability import run_condition
import argparse
cfg = argparse.Namespace(n_train=3000, n_probe=600, n_test=800, hidden=64, epochs=250)

print(f'{"corruption":12s} {"gamma":>7s} {"detect AUROC":>13s} {"gating benefit":>15s}')
for mode in ['zeroed', 'resampled']:
    r = run_condition(inst, f=0.6, corruption=mode, cfg=cfg, seed=0)
    print(f'{mode:12s} {r["gamma"]:7.1f} {r["detection_auroc"]:13.3f} {r["selective_gain"]:15.3f}')

You should see the punchline already: at the **same** `γ`, *zeroed* is detected
(AUROC ≈ 0.9) and gating helps a lot, while *resampled* sits at chance (≈ 0.5) and
gating gains nothing. Same error size, opposite outcome.

## 4. The full sweep and the four-panel figure

Now sweep `γ` and reproduce the headline figure. (This is the slow cell — ~1–2
minutes; it trains a fresh small network at each gap for 3 seeds.)

In [ ]:
from toy_identifiability import sweep, risk_coverage_demo, resampled_scores, plot_figure
cfg = argparse.Namespace(
    seeds=[0,1,2], k_core=8, k_periph=8, sigma=0.1,
    n_train=3000, n_probe=600, n_test=800, hidden=64, epochs=250,
    n_f=9, f_min=0.02, f_max=0.85, demo_f=0.6,
)
sw = sweep(inst, cfg)
rc = risk_coverage_demo(inst, cfg, seed=0)
rs = resampled_scores(inst, cfg, seed=0)

out = Path('results/toy_identifiability'); out.mkdir(parents=True, exist_ok=True)
plot_figure(sw, rc, rs, out)
from IPython.display import Image
Image(str(out / 'toy_gating.png'))

### Reading the figure

- **A — Detecting unanswerability.** Two flat lines. *Zeroed* stays near 0.9 at
  every `γ`; *resampled* stays at chance. Detection does **not** track `γ`.
- **B — Benefiting from abstention.** *Zeroed* captures ~85–90% of the oracle's
  benefit; *resampled* captures ~0. Again flat in `γ`.
- **C — Risk–coverage: the model restraining itself.** This is *the* gating plot.
  As coverage drops (we answer fewer inputs, abstaining on the least-confident),
  the retained error falls. The probe (blue) hugs the oracle (violet), far below
  random (grey). That gap is the value of gating.
- **D — Probe scores.** Answerable (green) and zeroed-unanswerable (blue) separate
  cleanly; resampled-unanswerable (yellow) sits right on top of answerable — the
  probe literally cannot tell them apart.

**The one sentence to remember:** *whether gating works is decided by whether the
unanswerability is visible in the input — not by how wrong the answer would be.*
A deterministic forward pass on an in-distribution input produces in-distribution
activations, so no probe on that single pass can flag it, no matter how large the
error. This is exactly why, in the physiological benchmark, detecting *missing
channels* was easy but detecting *subtly-wrong-but-complete* signals collapsed to
chance.

## 5. Geometry: how is 'unanswerable' arranged inside the network?

For the detectable (zeroed) case, *where* does the answerability signal live in
activation space? We measure three things:

- **Rank** — erase the top linear probe direction, refit, repeat. If one removal
  kills detection, the signal is a single direction (rank-1). If it takes many, the
  signal is spread out.
- **Concentration** — how much of the full detection does the single strongest
  direction alone capture?
- **A 2D picture** — project activations onto the answerability axis and the top
  orthogonal direction.

In [ ]:
from toy_geometry import plot_geometry
geo = plot_geometry(inst, cfg, out)
r = geo['rank_reference']
print(f"full-probe AUROC = {r['full_auroc']:.3f}")
print(f"single best direction alone = {r['top1_auroc']:.3f}  ({r['concentration']:.0%} of the full signal)")
Image(str(out / 'toy_geometry.png'))

### What the geometry says

The answerability signal is **approximately low-rank but not rank-1**: one
dominant 'answerability axis' carries ~88% of the detection, yet erasing it doesn't
kill detection — the remainder is smeared across roughly the dimensionality of the
corrupted block (here 8). That makes sense: *'is anything missing?'* collapses
mostly onto one direction, but *'which values are missing?'* needs several. The
concentration rises slightly with `γ` (Panel C): a bigger gap makes the signal a
little cleaner.

This is the honest answer to 'is there a single "I don't know" direction?' —
**mostly, but with a tail**, and the tail is tied to the structure of what went
missing.

## 6. Takeaways

1. **Gating provably works** when unanswerability is visible in the input — the
   probe tracks the oracle (Panel C).
2. **Gating provably cannot work** from a single forward pass when the corruption
   keeps the input in-distribution — regardless of how large the error is.
3. **The internal 'I don't know' signal is approximately low-rank** — one dominant
   direction plus a structured tail.

The practical lesson for the physiological benchmark (and any real deployment):
detecting *missing inputs* is easy and worth doing; catching *plausible-but-wrong*
inputs needs more than one deterministic forward pass — ensembles, test-time
perturbations, or access to the missing variable itself.

See `analysis/THEORY.md` for the formal version and `analysis/SUMMARY.md` for how
this connects to the two physiological laws.